## Load libraries

In [1]:
!pip install -q -U transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00


## Imports

In [2]:
import torch
from transformers import AutoTokenizer, LongT5ForConditionalGeneration

## Get model and tokenizer

In [3]:
model_name = "google/long-t5-tglobal-base"


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = LongT5ForConditionalGeneration.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

## Build prompt

In [9]:
# from IPython.display import display, Markdown

def summarize_long_t5(text, max_words=0):
    # Long-T5 can handle up to 16,384 tokens
    inputs = tokenizer("summarize: " + text, return_tensors="pt", truncation=True, max_length=16384).to(model.device)

    if max_words > 0:
      # We use max_new_tokens to approximate the word cap
      # 1 word is roughly 1.3 to 1.5 tokens
      max_tokens = int(max_words * 1.5)

    else:
      max_tokens = 4000

    output_ids = model.generate(
          inputs["input_ids"],
          max_new_tokens=max_tokens,
          num_beams=4,
          length_penalty=2.0,
          early_stopping=True
      )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# def summarize(text, max_words = 0):
#     # Build the prompt string
#     prompt = "<start_of_turn>user\n"
#     prompt += "SYSTEM: You are a summarization expert. You return ONLY the summarized text adhering to any and all requirements/constraints specified.\n\n"
#     prompt += f"Summarize the following text."
#     if (max_words > 0):
#         prompt += f" The summary should be at most {max_words} words."

#     prompt += f"\n\n{text}<end_of_turn>\n<start_of_turn>model\n"

#     # Visual
#     display(Markdown("---"))
#     display(Markdown(f"**The following text is being sent to the model:**\n\n```text\n{prompt}\n```"))

#     # Execute
#     inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=1024,
#             temperature=0.01,
#             do_sample=False
#         )

#     # Extract and format response
#     response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

#     display(Markdown(f"**MODEL RESPONSE**\n{response}"))
#     return response

## Sample it

In [10]:
text = "EPIDIOLEX- cannabidiol solution Jazz Pharmaceuticals, Inc. ---------- HIGHLIGHTS OF PRESCRIBING INFORMATION These highlights do not include all the information needed to use EPIDIOLEX® safely and effectively. See full prescribing information for EPIDIOLEX. EPIDIOLEX® (cannabidiol) oral solution Initial U.S. Approval: 2018 INDICATIONS AND USAGE EPIDIOLEX is indicated for the treatment of seizures associated with Lennox-Gastaut syndrome, Dravet syndrome, or tuberous sclerosis complex in patients 1 year of age and older (1) DOSAGE AND ADMINISTRATION ••Obtain serum transaminases (ALT and AST) and total bilirubin levels in all patients prior to starting treatment. (2.1, 5.1) See Full Prescribing Information for titration. (2.2, 2.3) Seizures Associated with Lennox-Gastaut Syndrome or Dravet Syndrome ••The recommended starting dosage is 2.5 mg/kg by mouth twice daily (5 mg/kg/day). After one week, the dosage can be increased to a maintenance dosage of 5 mg/kg twice daily (10 mg/kg/day). (2.2) Based on individual clinical response and tolerability, EPIDIOLEX can be increased up to a maximum recommended maintenance dosage of 10 mg/kg twice daily (20 mg/kg/day). Seizures Associated with Tuberous Sclerosis Complex •The recommended starting dosage is 2.5 mg/kg by mouth twice daily (5 mg/kg/day). Increase the dose weekly by 2.5 mg/kg twice daily (5 mg/kg/day), as tolerated, to a recommended maintenance dosage of 12.5 mg/kg twice daily (25 mg/kg/day). (2.3) Patients with Impaired Hepatic Function • Dosage adjustment is recommended for patients with moderate or severe hepatic impairment. (2.6, 8.6) DOSAGE FORMS AND STRENGTHS Oral solution: 100 mg/mL (3) CONTRAINDICATIONS Hypersensitivity to cannabidiol or any of the ingredients in EPIDIOLEX (4) WARNINGS AND PRECAUTIONS •••••Hepatic Injury: EPIDIOLEX can cause transaminase elevations. Concomitant use of valproate and higher doses of EPIDIOLEX increase the risk of transaminase elevations. See Full Prescribing Information for serum transaminase and bilirubin monitoring recommendations. (5.1) Somnolence and Sedation: Monitor for somnolence and sedation and advise patients not to drive or operate machinery until they have gained sufficient experience on EPIDIOLEX. (5.2) Suicidal Behavior and Ideation: Monitor patients for suicidal behavior and thoughts. (5.3) Hypersensitivity Reactions: Advise patients to seek immediate medical care. Discontinue and do not restart EPIDIOLEX if hypersensitivity occurs. (5.4) Withdrawal of Antiepileptic Drugs: EPIDIOLEX should be gradually withdrawn to minimize the risk of increased seizure frequency and status epilepticus. (5.5) ADVERSE REACTIONS The most common adverse reactions (10% or more for EPIDIOLEX and greater than placebo) in patients with Lennox-Gastaut syndrome or Dravet syndrome are: somnolence; decreased appetite; diarrhea; transaminase elevations; fatigue, malaise, and asthenia; rash; insomnia, sleep disorder, and poor quality sleep; and infections. (6.1) The most common adverse reactions (10% or more for EPIDIOLEX and greater than placebo) in patients The most common adverse reactions (10% or more for EPIDIOLEX and greater than placebo) in patients with tuberous sclerosis complex are: diarrhea; transaminase elevations; decreased appetite; somnolence; pyrexia; and vomiting. (6.1) To report SUSPECTED ADVERSE REACTIONS, contact Jazz Pharmaceuticals, Inc. at 1-800- 520-5568 or FDA at 1-800-FDA-1088 or www.fda.gov/medwatch. DRUG INTERACTIONS ••••Strong inducer of CYP3A4 or CYP2C19: Consider dose increase of EPIDIOLEX. (7.1) Consider a dose reduction of substrates of CYP1A2, CYP2C8, UGT1A9, and orally administered P-gp substrates. (7.2) A lower starting dose of orally administered everolimus is recommended. (7.2) Consider dose modification of CYP2B6 or CYP2C19 substrates. (7.2) USE IN SPECIFIC POPULATIONS Pregnancy: Based on animal data, may cause fetal harm. (8.1) See 17 for PATIENT COUNSELING INFORMATION and Medication Guide. Revised: 7/2025 FULL PRESCRIBING INFORMATION: CONTENTS* 1 INDICATIONS AND USAGE 2 DOSAGE AND ADMINISTRATION 2.1 Assessments Prior to Initiating EPIDIOLEX 2.2 Dosing for Seizures Associated with Lennox-Gastaut Syndrome or Dravet Syndrome 2.3 Dosing for Seizures Associated with Tuberous Sclerosis Complex 2.4 Administration Instructions 2.5 Discontinuation of EPIDIOLEX 2.6 Patients with Hepatic Impairment 3 DOSAGE FORMS AND STRENGTHS 4 CONTRAINDICATIONS 5 WARNINGS AND PRECAUTIONS 5.1 Hepatic Injury 5.2 Somnolence and Sedation 5.3 Suicidal Behavior and Ideation 5.4 Hypersensitivity Reactions 5.5 Withdrawal of Antiepileptic Drugs (AEDs) 6 ADVERSE REACTIONS 6.1 Clinical Trials Experience 7 DRUG INTERACTIONS 7.1 Effect of Other Drugs on EPIDIOLEX 7.2 Effect of EPIDIOLEX on Other Drugs 7.3 Concomitant Use of EPIDIOLEX and Valproate 7.4 CNS Depressants and Alcohol 8 USE IN SPECIFIC POPULATIONS 8.1 Pregnancy 8.2 Lactation 8.4 Pediatric Use 8.5 Geriatric Use 8.6 Hepatic Impairment 9 DRUG ABUSE AND DEPENDENCE 9.1 Controlled Substance 9.2 Abuse 9.3 Dependence 11 DESCRIPTION 12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Action 12.2 Pharmacodynamics 12.3 Pharmacokinetics 13 NONCLINICAL TOXICOLOGY 13.1 Carcinogenesis and Mutagenesis 14 CLINICAL STUDIES 14.1 Lennox–Gastaut Syndrome 14.2 Dravet Syndrome 14.3 Tuberous Sclerosis Complex 16 HOW SUPPLIED/STORAGE AND HANDLING 16.1 How Supplied 16.2 Storage and Handling 17 PATIENT COUNSELING INFORMATION * Sections or subsections omitted from the full prescribing information are not listed. FULL PRESCRIBING INFORMATION 1 INDICATIONS AND USAGE EPIDIOLEX is indicated for the treatment of seizures associated with Lennox-Gastaut syndrome (LGS), Dravet syndrome (DS), or tuberous sclerosis complex (TSC) in patients 1 year of age and older. 2 DOSAGE AND ADMINISTRATION 2.1 Assessments Prior to Initiating EPIDIOLEX Because of the risk of hepatocellular injury, obtain serum transaminases (ALT and AST) and total bilirubin levels in all patients prior to starting treatment with EPIDIOLEX [see Warnings and Precautions (5.1)]. 2.2 Dosing for Seizures Associated with Lennox-Gastaut Syndrome or Dravet Syndrome •••The starting dosage is 2.5 mg/kg by mouth twice daily (5 mg/kg/day). After one week, the dosage can be increased to a maintenance dosage of 5 mg/kg twice daily (10 mg/kg/day). Patients who are tolerating EPIDIOLEX at 5 mg/kg twice daily and require further reduction of seizures may benefit from a dosage increase up to a maximum recommended maintenance dosage of 10 mg/kg twice daily (20 mg/kg/day), in weekly increments of 2.5 mg/kg twice daily (5 mg/kg/day), as tolerated. For patients in whom a more rapid titration from 10 mg/kg/day to 20 mg/kg/day is warranted, the dosage may be increased no more frequently than every other day. Administration of the 20 mg/kg/day dosage resulted in somewhat greater reductions in seizure rates than the recommended maintenance dosage of 10 mg/kg/day, but with an increase in adverse reactions. 2.3 Dosing for Seizures Associated with Tuberous Sclerosis Complex •••The starting dosage is 2.5 mg/kg by mouth twice daily (5 mg/kg/day). Increase the dose in weekly increments of 2.5 mg/kg twice daily (5 mg/kg/day), as tolerated, to a recommended maintenance dosage of 12.5 mg/kg twice daily (25 mg/kg/day). For patients in whom a more rapid titration to 25 mg/kg/day is warranted, the dosage may be increased no more frequently than every other day. The effectiveness of doses lower than 12.5 mg/kg twice daily has not been studied in patients with TSC." # @param {type:"string", placeholder:"Text to summarize"}
max_words = 0 # @param {type:"integer", placeholder:"Use zero for no limit"}

summarization = summarize_long_t5(text, max_words)

from IPython.display import display, Markdown
display(Markdown(f"**MODEL RESPONSE**\n{summarization}"))

**MODEL RESPONSE**
Revised: 7/2025 FULL PRESCRIBING INFORMATION: CONTENTS* 1 INDICATIONS AND USAGE 2 DOSAGE AND ADMINISTRATION 2.1 Assessments Prior to Initiating EPIDIOLEX 2.2 Dosing for Seizures Associated with Lennox-Gastaut Syndrome or Dravet Syndrome 2.3 Dosing for Seizures Associated with Tuberous Sclerosis Complex 2.4 Administration Instructions 2.5 Discontinuation of EPIDIOLEX 2.6 Patients with Hepatic Impairment 3 DOSAGE FORMS AND STRENGTHS 4 CONTRAINDICATIONS 5 WARNINGS AND PRECAUTIONS 5.1 Hepatic Injury 5.2 Somnolence and Sedation 5.3 Suicidal Behavior and Ideation 5.4 Hypersensitivity Reactions 5.5 Withdrawal of Antiepileptic Drugs (AEDs) 6 ADVERSE REACTIONS 6.1 Clinical Trials Experience 7 DRUG INTERACTIONS 7.1 Effect of Other Drugs on EPIDIOLEX 7.2 Effect of EPIDIOLEX on Other Drugs 7.3 Concomitant Use of EPIDIOLEX and Valproate 7.4 CNS Depressants and Alcohol 8 USE IN SPECIFIC POPULATIONS 8.1 Pregnancy 8.2 Lactation 8.4 Pediatric Use 8.5 Geriatric Use 8.6 Hepatic Impairment 9 DRUG ABUSE AND DEPENDENCE 9.1 Controlled Substance 9.2 Abuse 9.3 Dependence 11 DESCRIPTION 12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Action 12.2 Pharmacodynamics 12.3 Pharmacokinetics 13 NONCLINICAL TOXICOLOGY 13.1 Carcinogenesis and Mutagenesis 14 CLINICAL STUDIES 14.1 Lennox–Gastaut Syndrome 14.2 Dravet Syndrome 14.3 Tuberous Sclerosis Complex 16 HOW SUPPLIED/STORAGE AND HANDLING 16.1 How Supplied 16.2 Storage and Handling 17 PATIENT COUNSELING INFORMATION * Sections or subsections omitted from the full prescribing information are not listed. Patients who are tolerating EPIDIOLEX at 5 mg/kg twice daily and require further reduction of seizures may benefit from a dosage increase up to a maximum recommended maintenance dosage of 10 mg/kg twice daily (20 mg/kg/day), in weekly increments of 2.5 mg/kg twice daily (5 mg/kg/day), as tolerated.